# GRPO training: per-step cost on Colab

Times one training step of our three configs by calling `train/run_train.py`. The training loop
itself is not reimplemented here; the benchmark helpers live in `train/benchmark.py`.

| Config | Mode | B | G |
|---|---|---|---|
| `single_step_matched` | single | 16 | 8 |
| `single_token_matched` | single | 16 | 16 |
| `multi` | multi (solver + reviser) | 16 | 8 |

Outputs go to Google Drive (`OUTPUT_ROOT/benchmark/`) so they survive a disconnect:
`raw.jsonl` (every step), `step_cost.csv`, `step_cost.tex`, plus per-config results and logs.

**Runtime → Change runtime type → GPU.** An L4 or A100 is needed for bf16; on a T4 the
benchmark falls back to float16 and says so in every output. Total time: roughly 10 minutes of
setup plus 10–30 minutes per config, depending on the GPU.

After a disconnect or restart, rerun the settings cell and section 1, then continue: finished
configs are skipped unless `FORCE_RERUN = True`.

In [ ]:
# ---- Settings: the only values you should need to change ----
REPO_URL = "https://github.com/supmine/COMS4705_Project_Code.git"
BRANCH = "feature/training-loop"      # switch to "main" once the training loop is merged
OUTPUT_ROOT = "/content/drive/MyDrive/coms4705_runs"
N_EPOCHS = 1                          # for the compute estimate
EVAL_OVERHEAD = 0.1                   # extra fraction of time for dev evaluation and checkpoints
FORCE_RERUN = False                   # True: rerun configs whose results already exist

REPO_DIR = "/content/COMS4705_Project_Code"
BENCH_DIR = f"{OUTPUT_ROOT}/benchmark"
N_STEPS = 5                           # step 1 (warmup) is excluded from the medians

## 1. Setup

Checks the GPU and bf16 support, mounts Google Drive, clones the repo, and installs pinned
library versions. Takes about 5–10 minutes the first time and a few seconds after that.

If the GPU has no bf16 (compute capability below 8.0, e.g. a T4), the benchmark runs in
float16 (`--allow_fp16`) and records `dtype` in every output. Timings then differ from what an
L4/A100 would give and should not be reported as the bf16 cost.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                        "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
GPU_NAME, GPU_MEMORY, COMPUTE_CAP = [x.strip() for x in query.splitlines()[0].split(",")]
BF16_SUPPORTED = float(COMPUTE_CAP) >= 8.0
ALLOW_FP16 = not BF16_SUPPORTED
DTYPE = "bfloat16" if BF16_SUPPORTED else "float16"
print(f"GPU: {GPU_NAME}, {GPU_MEMORY}, compute capability {COMPUTE_CAP}")
print(f"bf16 supported: {BF16_SUPPORTED} -> dtype {DTYPE}")
if not BF16_SUPPORTED:
    print("\n" + "!" * 78 + f"\nWARNING: {GPU_NAME} has no bf16. Running in float16 (allow_fp16: true).\n"
          "These timings are NOT the bf16 cost; use an L4 or A100 for reported numbers.\n" + "!" * 78)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(BENCH_DIR, exist_ok=True)
print("outputs ->", BENCH_DIR)

In [ ]:
import os
import subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
os.chdir(REPO_DIR)
GIT_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("repo", REPO_DIR, "branch", BRANCH, "commit", GIT_COMMIT)

### Install pinned versions

vLLM pins torch exactly, so torch, vLLM, transformers, peft and datasets are installed together
at versions that satisfy each other's requirements:
torch 2.14.0, vLLM 0.33.0, transformers 5.20.2, peft 0.21.3, datasets 5.2.0 (+ math-verify 0.9.0).
Colab's preinstalled `torchao` 0.10 is removed: PEFT 0.21 refuses to load next to torchao < 0.16,
and nothing in this project uses it.

**A restart is required only when the install actually changes packages** (it replaces Colab's
torch). The cell then restarts the session by itself. After the restart, rerun the settings cell
and section 1 from the top: this cell will find the versions in place and skip the install.

In [ ]:
import os
import subprocess
from importlib.metadata import version, PackageNotFoundError

PINS = {"torch": "2.14.0", "vllm": "0.33.0", "transformers": "5.20.2", "peft": "0.21.3",
        "datasets": "5.2.0", "math-verify": "0.9.0"}

def installed(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None

# Colab ships torchao 0.10, and PEFT 0.21 refuses to load next to torchao < 0.16. Nothing here
# uses torchao, so remove it. No restart needed: tests and training run in fresh processes.
if installed("torchao"):
    print("removing torchao", installed("torchao"))
    subprocess.run(["pip", "uninstall", "-y", "-q", "torchao"], check=True)

missing = {p: v for p, v in PINS.items() if installed(p) != v}
if missing:
    print("installing:", missing)
    subprocess.run(["pip", "install", "-q"] + [f"{p}=={v}" for p, v in PINS.items()]
                   + ["pyyaml", "matplotlib", "pytest"], check=True)
    print("\nPackages changed: restarting the session. Then rerun from the settings cell.")
    os.kill(os.getpid(), 9)   # Colab restarts the runtime; files and Drive stay
else:
    print("pinned versions already installed; no restart needed")
for p in PINS:
    print(f"{p:14s} {installed(p)}")

## 2. Unit tests

Runs the repo's tests (answer checking, data split, prompts) and the trainer tests in `train/`
(loss, masking, adapters, benchmark summary). Stops the notebook if anything fails. Takes about
2–4 minutes (it downloads GSM8K, MATH and the Qwen tokenizer).

In [ ]:
import os
import subprocess
os.chdir(REPO_DIR)
tests = subprocess.run(["python", "-m", "pytest", "tests/", "train/", "-q"], capture_output=True, text=True)
print(tests.stdout[-4000:], tests.stderr[-2000:])
if tests.returncode != 0:
    raise RuntimeError("Unit tests failed: fix them before timing anything.")

## 3. Timing benchmark

For each config: 5 training steps (B = 16 GSM8K train problems per step) in a fresh process via
`subprocess`, with dev evaluation and checkpoints turned off. Step 1 (model loading, vLLM warmup)
is excluded; the summary is the median of steps 2–5 for total, generation and training seconds,
generated and prefill tokens, and peak GPU memory.

If a config runs out of memory it is retried with smaller micro-batches (gradient accumulation,
same optimizer steps: default packing → 4 → 1 sequences); the micro-batch size used is recorded.
If it still fails it is recorded as `OOM` and the next config runs.

Expect roughly 10–30 minutes per config (model and vLLM start-up is a few minutes of that).
Each finished config is saved to `benchmark/results/<config>.json` and skipped on a rerun unless
`FORCE_RERUN = True`.

In [ ]:
import os
import sys
import pandas as pd
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
from train import benchmark as bm

results = [bm.run_config(c, BENCH_DIR, max_steps=N_STEPS, allow_fp16=ALLOW_FP16,
                         force=FORCE_RERUN, cwd=REPO_DIR) for c in bm.CONFIGS]
rows = [bm.summarize(r) for r in results]
bm.write_raw(results, f"{BENCH_DIR}/raw.jsonl")
bm.write_csv(rows, f"{BENCH_DIR}/step_cost.csv")
print("saved", f"{BENCH_DIR}/raw.jsonl", "and", f"{BENCH_DIR}/step_cost.csv")
pd.DataFrame(rows)[["run", "status", "G", "B", "dtype", "micro_batch_size", "step_seconds",
                    "step_gen_seconds", "step_train_seconds", "step_gen_tokens",
                    "step_prefill_tokens", "peak_mem_gb", "device_mem_used_gb"]]

## 4. Results

Writes the LaTeX table, checks that the token-matched and multi-agent runs generate about twice
the tokens of the step-matched run, and estimates the compute for full training runs. Takes
seconds.

`peak_mem_gb` is `torch.cuda.max_memory_allocated` of the training process; vLLM's engine runs in
its own process, so its memory (about `gpu_memory_utilization` = 35% of the GPU) is not in that
number. `device_mem_used_gb` in the CSV is the whole GPU, vLLM included.

In [ ]:
gpus = {r["gpu"] for r in results if r.get("gpu")} or {GPU_NAME}
dtypes = {r["dtype"] for r in results if r.get("dtype")} or {DTYPE}
B = next((r["batch_problems"] for r in results if r.get("batch_problems")), 16)
tex = bm.write_tex(rows, f"{BENCH_DIR}/step_cost.tex", B=B, gpu=", ".join(sorted(gpus)),
                   dtype=", ".join(sorted(dtypes)))
print(tex)
print("\nGenerated tokens per step relative to single_step_matched (expected ~2 for the other two):")
for config, ratio in bm.token_ratios(rows).items():
    print(f"  {config:22s} {ratio:.2f}")

In [ ]:
from common.data import split_spec
TRAIN_SIZE = split_spec()["sizes"]["train"]     # GSM8K train minus the 500 dev problems (6,973)

estimate, total = bm.compute_estimate(rows, TRAIN_SIZE, B, N_EPOCHS, EVAL_OVERHEAD)
print(f"TRAIN_SIZE={TRAIN_SIZE}, B={B}, N_EPOCHS={N_EPOCHS}, EVAL_OVERHEAD={EVAL_OVERHEAD}")
for e in estimate:
    hours = "OOM / not measured" if e["hours"] is None else f"{e['hours']:.1f} h"
    s = "--" if e["seconds_per_step"] is None else f"{e['seconds_per_step']:.1f}"
    print(f"  {e['run']:30s} {e['steps_per_epoch']} steps/epoch x {N_EPOCHS} epoch(s), "
          f"{s} s/step -> {hours}")
print(f"Total for the three runs: {'n/a (a config did not finish)' if total is None else f'{total:.1f} h'}")
print(f"Times measured on this Colab GPU: {', '.join(sorted(gpus))} ({', '.join(sorted(dtypes))}), "
      f"repo commit {GIT_COMMIT[:7]}.")